In [0]:
# Import game name mapping from CSV
import pandas as pd
import re 
from collections.abc import Mapping, Sequence
from typing import Any
from typing import Any
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from operator import or_

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


import numpy as np
import pandas as pd

# paths etc
eilers_performance_path="development_011_bronze_core.external_data.eilers_landbased_performance_raw"
# =========================
# Configuration
# =========================
# load params
FILTERS = {
    "Region_Name": "U.S. & Canada",
    "Game_Classification": "CLASS 3",
}
#Standardize params
rename_map_perf={}
lower_case_perf=True
standardize_perf=True
#fix_missing_values params
CLEANING_CONFIG = {
    "date_cols": [
        "yearmonth",
        "game_release_date",
        "cabinet_release_date",
    ],

    "date_format": None,

    "missing_tokens": [
        "-",
        "--",
        "",
        " ",
        "other",
        "unknown",
        "N/A",
        "n/a",
        "NULL",
        "null",
        "(Blank)",
        "#",
        "Not Provided",
    ],

    "numeric_cols": [
        "game_denom",
        "avg_days_terminal",
        "avg_days_game_theme",
        "no_of_slots",
        "avg_coin_in_index_vs_house",
        "avg_coin_in_index_vs_zone",
        "total_avg_theo_net_win_index_vs_house",
        "total_avg_theo_net_win_index_vs_zone",
        "total_avg_coin_in_index_vs_house",
        "total_avg_coin_in_index_vs_zone",
        "avg_average_bet",
        "casinos",
        "avg_games_played_index",
        "theo_win_vs_house",
        "theo_win_vs_zone",
        "average_bet_index_same_denom",
        "average_bet_index_same_type_same_denom",
        "coin_in_vs_house_same_type",
        "theo_win_vs_house_same_type_same_denom",
        "utilization_index_same_type_same_denom",
        "no_of_casinos",
        "avg_theo_net_win_index_vs_house",
        "avg_theo_net_win_index_vs_zone",
        "casino_count",
        "avg_state_casino_count",
    ],

    "string_cols": [
        "supplier",
        "parent_supplier",
        "game_name",
        "slot_cabinet_name",
        "state",
    ],

    "lower_case": True,

    # Add column-specific corrections here.
    # Mapping keys should normally be lowercase when lower_case=True.
    "typo_maps": {
        # Example:
        # "supplier": {
        #     "igt gaming": "igt",
        #     "international game technology": "igt",
        # },
        #
        # "state": {
        #     "neveda": "nevada",
        # },
    },
}
#fix_complementary_columns
complementary_columns = {
    "casinos": ["casino_count"],
}

drop_complementary_fallbacks = False
#fix_supplier_names
supplier_map = {
    # Light & Wonder / legacy SG ecosystem
    "lightning gaming": "light & wonder",
    "scientific games": "light & wonder",
    "sg gaming": "light & wonder",
    "sgi": "light & wonder",

    # Merkur
    "merkur": "merkur gaming",
    "merkur-gaming": "merkur gaming",

    # TCS John Huxley
    "tcsjohnhuxley": "tcs john huxley",

    # Cosmetic normalizations
    "bluberi gaming": "bluberi",

    # Common typo / formatting cleanup
    "light and wonder": "light & wonder",
    "light&wonder": "light & wonder",
    "l&w": "light & wonder",

    "aristocrat gaming": "aristocrat",
    "aristocrat technologies": "aristocrat",

    "igt gaming": "igt",
    "international game technology": "igt",
    "itt": "igt",

    "konami gaming": "konami",

    "ags gaming": "ags",
    "american gaming systems": "ags",

    "incredible tech": "incredible technologies",

    "everi gaming": "everi",

    "gaming arts llc": "gaming arts",

    "euro game technology": "euro games technology",

    "apex": "apex gaming",

    "inspired gaming": "inspired",

    "rocket gaming": "rocket gaming systems",

    "playsynergy": "play synergy",

    "synergyblue": "synergy blue",

    "winsystems": "win systems",

    "interblock gaming": "interblock",

    "alfastreet gaming": "alfastreet",

    "spintec gaming": "spintec",

    "fbm gaming": "fbm",

    "aruze gaming": "aruze",

    "sega sammy creation": "sega sammy",
}
# apply_ownership_mapping
ownership_mapping = {

    "input_col": "own_status",
    "output_col": "own_status",

    "owned_exact": [
        "owned",
        "own",
        "o",
        "y",
        "yes",
        "true",
    ],

    "leased_exact": [
        "leased",
        "lease",
        "n",
        "no",
        "false",
        "not owned",
    ],

    "owned_contains": [
        "own",
    ],

    "leased_contains": [
        "lease",
        "%",
        "ci-",
        "daily",
        "per day",
        "net win",
        "cap",
    ],
}

In [0]:



def apply_ownership_mapping(
    df: DataFrame,
    config: dict | None = None,
) -> DataFrame:
    if not config:
        return df

    input_col = config["input_col"]
    output_col = config.get("output_col", input_col)

    x = F.lower(F.trim(F.col(input_col).cast("string")))

    def contains_any(values):
        return reduce(or_, [x.contains(v.lower()) for v in values], F.lit(False))

    owned_exact = [v.lower() for v in config.get("owned_exact", [])]
    leased_exact = [v.lower() for v in config.get("leased_exact", [])]

    return df.withColumn(
        output_col,
        F.when(x.isin(owned_exact), "owned")
        .when(x.isin(leased_exact), "leased")
        .when(contains_any(config.get("leased_contains", [])), "leased")
        .when(
            contains_any(config.get("owned_contains", []))
            & ~x.contains("lease"),
            "owned",
        )
        .otherwise(F.lit(None).cast("string")),
    )


In [0]:
#cleaning functions tested
def standardize_column_name(col: str, lower_case: bool = True) -> str:
    """Return the default normalized form used by normalize_columns."""

    col = str(col).strip()
    if lower_case:
        col = col.lower()
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[-/]+", "_", col)
    col = re.sub(r"[^\w_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")
def normalize_columns(
    df,
    rename_map: dict | None = None,
    standardize: bool = True,
    lower_case: bool = True,
):
    """
    Normalize Spark DataFrame column names.

    Order:
    1. Apply explicit rename_map
    2. Optionally standardize column names
    3. Check for duplicate columns after normalization
    """

    cols = df.columns

    if rename_map:
        cols = [rename_map.get(col, col) for col in cols]

    if standardize:
        new_cols = []
        for col in cols:
            col = str(col).strip()
            col = standardize_column_name(col, lower_case=lower_case)
            new_cols.append(col)
        cols = new_cols

    duplicated = [col for col in set(cols) if cols.count(col) > 1]
    if duplicated:
        raise ValueError(
            f"Duplicate columns after normalization: {duplicated}. "
            "Fix rename_map or column cleanup rules."
        )

    for old, new in zip(df.columns, cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

    return df
    #-----------------------
    

def _quote_spark_column(column_name: str) -> str:
    """
    Safely quote a Spark column name.

    This allows columns containing spaces, punctuation, or reserved words
    to be used inside Spark SQL expressions.
    """
    escaped_name = column_name.replace("`", "``")
    return f"`{escaped_name}`"


def fix_missing_values_spark(
    df: DataFrame,
    numeric_cols: list[str] | None = None,
    typo_maps: dict[str, dict[Any, Any]] | None = None,
    missing_tokens: list[str] | None = None,
    string_cols: list[str] | None = None,
    date_cols: list[str] | None = None,
    date_format: str | None = None,
    lower_case: bool = True,
) -> DataFrame:
    """
    Fix common raw-data issues in a Spark DataFrame.

    Operations:
    - Replace configured missing-value tokens with NULL
    - Remove commas and percent signs from numeric columns
    - Convert invalid numeric values to NULL
    - Trim and optionally lowercase configured string columns
    - Convert purely numeric values in configured string columns to NULL
    - Convert configured columns to Spark date type
    - Apply categorical typo mappings

    Notes:
    - Spark DataFrames are immutable, so every transformation returns a
      new DataFrame.
    - Numeric columns are converted to double.
    - `try_cast` is used so invalid values become NULL rather than causing
      an error when ANSI mode is enabled.
    """

    # ------------------------------------------------------------------
    # Default missing-value tokens
    # ------------------------------------------------------------------
    if missing_tokens is None:
        missing_tokens = [
            "-",
            "--",
            "",
            " ",
            "N/A",
            "n/a",
            "NULL",
            "null",
            "(Blank)",
            "#",
        ]

    # Replace exact missing-token matches across the DataFrame.
    # String replacements only affect compatible string columns.
    df = df.replace(
        to_replace=list(missing_tokens),
        value=None,
    )

    # ------------------------------------------------------------------
    # Numeric cleanup
    # ------------------------------------------------------------------
    if numeric_cols:
        for column_name in numeric_cols:
            if column_name not in df.columns:
                continue

            quoted_column = _quote_spark_column(column_name)

            # Match the Pandas logic:
            # 1. Cast to string
            # 2. Trim whitespace
            # 3. Remove commas
            # 4. Remove percent signs
            # 5. Convert to double
            #
            # try_cast changes invalid numeric values to NULL.
            numeric_expression = f"""
                try_cast(
                    regexp_replace(
                        regexp_replace(
                            trim(cast({quoted_column} AS string)),
                            ',',
                            ''
                        ),
                        '%',
                        ''
                    )
                    AS double
                )
            """

            df = df.withColumn(
                column_name,
                F.expr(numeric_expression),
            )

    # ------------------------------------------------------------------
    # String cleanup
    # ------------------------------------------------------------------
    if string_cols:
        numeric_string_pattern = r"^\d+(\.\d+)?$"

        for column_name in string_cols:
            if column_name not in df.columns:
                continue

            # Convert to string and remove leading/trailing whitespace.
            cleaned_value = F.trim(
                F.col(column_name).cast("string")
            )

            # Optionally standardize text to lowercase.
            if lower_case:
                cleaned_value = F.lower(cleaned_value)

            # Values such as "123" or "123.45" are invalid for these
            # categorical/string columns and are converted to NULL.
            df = df.withColumn(
                column_name,
                F.when(
                    cleaned_value.rlike(numeric_string_pattern),
                    F.lit(None).cast("string"),
                ).otherwise(cleaned_value),
            )

    # ------------------------------------------------------------------
    # Date conversion
    # ------------------------------------------------------------------
    if date_cols:
        for column_name in date_cols:
            if column_name not in df.columns:
                continue

            quoted_column = _quote_spark_column(column_name)

            if date_format:
                # Escape single quotes in case the supplied format contains
                # literal text.
                safe_date_format = date_format.replace("'", "''")

                date_expression = f"""
                    to_date(
                        try_to_timestamp(
                            {quoted_column},
                            '{safe_date_format}'
                        )
                    )
                """
            else:
                # Attempt Spark's normal timestamp parsing.
                # Invalid values become NULL.
                date_expression = f"""
                    to_date(
                        try_cast({quoted_column} AS timestamp)
                    )
                """

            df = df.withColumn(
                column_name,
                F.expr(date_expression),
            )

    # ------------------------------------------------------------------
    # Typo and categorical-value corrections
    # ------------------------------------------------------------------
    if typo_maps:
        for column_name, mapping in typo_maps.items():
            if column_name not in df.columns:
                continue

            df = df.replace(
                to_replace=dict(mapping),
                subset=[column_name],
            )

    return df
    ##--------- complementary columns
def fix_complementary_columns(
    df: DataFrame,
    column_groups: dict[str, list[str]],
    drop_fallbacks: bool = False,
) -> DataFrame:

    for primary_col, fallback_cols in column_groups.items():
        existing_cols = [
            col_name
            for col_name in [primary_col, *fallback_cols]
            if col_name in df.columns
        ]

        if not existing_cols:
            continue

        df = df.withColumn(
            primary_col,
            F.coalesce(*[F.col(col_name) for col_name in existing_cols]),
        )

        if drop_fallbacks:
            df = df.drop(
                *[
                    col_name
                    for col_name in fallback_cols
                    if col_name in df.columns
                ]
            )

    return df
#--------------- fixed supplier:
def fix_supplier_names_spark(
    df: DataFrame,
    supplier_col: str,
    supplier_map: Mapping[str, str],
    *,
    lower_case: bool = True,
    verbose: bool = True,
) -> DataFrame:
    """
    Standardize supplier names in a Spark DataFrame using a mapping.

    Operations:
    - Preserve NULL values
    - Cast supplier values to strings
    - Trim leading/trailing whitespace
    - Optionally lowercase values before mapping
    - Replace matching supplier names without using a Python UDF

    Example
    -------
    supplier_map = {
        "igt": "IGT",
        "igt canada": "IGT",
        "aruze gaming": "Aruze",
        "ags": "AGS",
    }

    df = fix_supplier_names_spark(
        df,
        supplier_col="supplier_name",
        supplier_map=supplier_map,
    )
    """

    if supplier_col not in df.columns:
        raise ValueError(
            f"Column {supplier_col!r} was not found in the DataFrame."
        )

    # Normalize mapping keys to match how the source column is cleaned.
    normalized_map = {
        str(key).strip().lower() if lower_case else str(key).strip(): value
        for key, value in supplier_map.items()
    }

    original_value = F.col(supplier_col)

    cleaned_value = F.trim(original_value.cast("string"))

    if lower_case:
        cleaned_value = F.lower(cleaned_value)

    if normalized_map:
        mapping_expression = F.create_map(
            *[
                item
                for key, value in normalized_map.items()
                for item in (F.lit(key), F.lit(value))
            ]
        )

        mapped_value = F.element_at(mapping_expression, cleaned_value)

        standardized_value = (
            F.when(original_value.isNull(), F.lit(None).cast("string"))
            .when(
                cleaned_value.isin(list(normalized_map)),
                mapped_value,
            )
            .otherwise(cleaned_value)
        )
    else:
        standardized_value = F.when(
            original_value.isNull(),
            F.lit(None).cast("string"),
        ).otherwise(cleaned_value)

    if verbose:
        changed_rows = df.filter(
            ~original_value.eqNullSafe(standardized_value)
        ).count()

        print(f"[fix_supplier_names_spark] updated {changed_rows:,} rows")

    return df.withColumn(supplier_col, standardized_value)


In [0]:
# load the data
from pyspark.sql.functions import col

df = spark.table(eilers_performance_path)

for column_name, filter_value in FILTERS.items():
    df = df.filter(col(column_name) == filter_value)


In [0]:
df=normalize_columns(df,rename_map=rename_map_perf,standardize=standardize_perf,lower_case=lower_case_perf)
df = fix_missing_values_spark(
    df,
    **CLEANING_CONFIG,
)
df = fix_complementary_columns(
    df,
    column_groups=complementary_columns,
    drop_fallbacks=drop_complementary_fallbacks,
)
df=fix_supplier_names_spark(
    df,
    supplier_col="supplier",
    supplier_map=supplier_map,
)
df= apply_ownership_mapping(df,ownership_mapping)

In [0]:
# to check the missing data stats
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType, DoubleType

total_rows = df.count()

missing_expressions = []

for field in df.schema.fields:
    column = F.col(field.name)

    # Standard NULL values
    missing_condition = column.isNull()

    # Include NaN for floating-point columns
    if isinstance(field.dataType, (FloatType, DoubleType)):
        missing_condition = missing_condition | F.isnan(column)

    # Include empty and whitespace-only strings
    if isinstance(field.dataType, StringType):
        missing_condition = missing_condition | (F.trim(column) == "")

    missing_expressions.append(
        F.sum(
            F.when(missing_condition, 1).otherwise(0)
        ).alias(field.name)
    )

missing_counts = df.agg(*missing_expressions).first().asDict()

missing_summary = spark.createDataFrame(
    [
        (
            field.name,
            field.dataType.simpleString(),
            missing_counts[field.name],
            round(
                missing_counts[field.name] / total_rows * 100,
                2,
            ) if total_rows else 0,
            total_rows - missing_counts[field.name],
        )
        for field in df.schema.fields
    ],
    [
        "column_name",
        "data_type",
        "missing_count",
        "missing_percent",
        "non_missing_count",
    ],
)

display(
    missing_summary.orderBy(
        F.desc("missing_percent")
    )
)

In [0]:
display(
    df.groupBy("Own_Status")
      .count()
      .orderBy(F.desc("count"))
)